# Embryonic data: covariance-derived T_s and hybrid mixing weight

hybrid_ts_soft_lincomb のODE/LinComb側の重み r(t) が diffusion timestep とともにどう変化するかを可視化する notebook。実行してもファイルは保存しない。

## Source code reused

- work/20260707_lincomb/configs/base.json and configs/hybrid_ts_soft_lincomb.json: 1000 diffusion steps, t_s=auto, gate_tau=20.0
- work/20260707_lincomb/scripts/common.py::load_experiment_config: base + experiment config merge
- work/20260707_lincomb/utils/regime_time.py::estimate_lambda_max_from_adata: AnnData inputから遺伝子共分散最大固有値 lambda_max を推定
- work/20260707_lincomb/utils/regime_time.py::estimate_ts_from_lambda: alpha_bar[t] * lambda_max ≈ 1 となる T_s を選択
- guided_diffusion/script_util.py::create_model_and_diffusion: 学習時と同じ diffusion schedule を構築
- ODE/ode_20260609_hybrid5x3.py::UnifiedODEMLHybrid._regime_ode_weight: r(t) = sigmoid((T_s - t) / gate_tau)
- work/20260707_lincomb/viz/plot_gate_diagnostics_0707.py::_save_gate_curve: gate curveの既存可視化ロジック

ここでの r(t) はODE/LinCombの重み、1-r(t) はCell_Unetの重み。共分散行列は直接 r を出すのではなく、lambda_max を通じて T_s を決めるために使われる。

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Local paths. The original h5ad-loading cell in this notebook is retained here.
REPO_ROOT = Path("/Users/cls-lab/Git/scDiffusionODE")
WORK_0707 = REPO_ROOT / "work/20260707_lincomb"
DATA_PATH = REPO_ROOT / "work/20260215_embryonic/data/Embryonic.h5ad"
CONFIG_PATH = WORK_0707 / "configs/hybrid_ts_soft_lincomb.json"

for import_path in (REPO_ROOT, WORK_0707):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

# Reused imports: work/20260707_lincomb/scripts/common.py
from scripts.common import load_experiment_config
# Reused imports: work/20260707_lincomb/utils/regime_time.py
from utils.regime_time import (
    estimate_lambda_max_from_adata,
    estimate_ts_from_lambda,
)
# Reused import: guided_diffusion/script_util.py
from guided_diffusion.script_util import (
    create_model_and_diffusion,
    model_and_diffusion_defaults,
)
# Reused method: ODE/ode_20260609_hybrid5x3.py
from ODE.ode_20260609_hybrid5x3 import UnifiedODEMLHybrid


: 

In [ ]:
# AnnData load: the same local Embryonic.h5ad used by the 0707 config.
adata = ad.read_h5ad(DATA_PATH)

# Reused exactly from work/20260707_lincomb/scripts/common.py::load_experiment_config
config = load_experiment_config(CONFIG_PATH)

# The tracked config has a former Linux path; use the local file actually loaded above.
config["data_dir"] = str(DATA_PATH)

print(f"AnnData shape: {adata.shape}")
print(f"diffusion_steps: {config['diffusion_steps']}")
print(f"t_s request: {config['t_s']!r}")
print(f"gate_tau: {config['gate_tau']}")

In [ ]:
# Reused from work/20260707_lincomb/scripts/train_0707.py::_resolve_ts.
# This estimates the largest eigenvalue of the gene covariance matrix in
# the same AnnData space used for model input.
ts_seed = config.get("ts_seed")
if ts_seed is None:
    ts_seed = config.get("seed", 0)

lambda_info = estimate_lambda_max_from_adata(
    adata,
    layer=config.get("ts_layer"),
    n_cells=int(config.get("ts_n_cells", 200_000)),
    seed=int(ts_seed),
    use_randomized_pca=True,
)

# Reused from guided_diffusion/script_util.py and train_0707.py.
# Only the diffusion schedule is used below; no trained checkpoint is loaded.
diffusion_args = model_and_diffusion_defaults()
diffusion_args["diffusion_steps"] = int(config["diffusion_steps"])
_, diffusion = create_model_and_diffusion(**diffusion_args)

# Reused exactly from train_0707.py::_resolve_ts.
ts_info = estimate_ts_from_lambda(
    diffusion.alphas_cumprod, lambda_info["lambda_max"]
)
T_s = int(ts_info["t_s"])

pd.Series({**lambda_info, **ts_info}, name="value")

In [ ]:
# Reused directly from ODE/ode_20260609_hybrid5x3.py::
# UnifiedODEMLHybrid._regime_ode_weight. The method only needs these four
# gate attributes, so no ODE weights or Cell_Unet checkpoint are required.
gate_state = SimpleNamespace(
    regime_gate_mode=config["regime_gate_mode"],
    regime_gate_type=config["regime_gate_type"],
    t_s=T_s,
    gate_tau=float(config["gate_tau"]),
)

t = np.arange(diffusion.num_timesteps, dtype=np.int64)
r_ode = UnifiedODEMLHybrid._regime_ode_weight(
    gate_state, torch.from_numpy(t), torch.device("cpu"), torch.float32
).squeeze(-1).numpy()
r_cellunet = 1.0 - r_ode
alpha_bar = np.asarray(diffusion.alphas_cumprod, dtype=np.float64)
covariance_score = alpha_bar * float(lambda_info["lambda_max"])

# Plot structure follows work/20260707_lincomb/viz/plot_gate_diagnostics_0707.py::_save_gate_curve.
fig, (ax_weight, ax_score) = plt.subplots(1, 2, figsize=(13, 4.2))

ax_weight.plot(t, r_ode, color="tab:blue", label="r(t): ODE / LinComb weight")
ax_weight.plot(t, r_cellunet, color="tab:orange", label="1-r(t): Cell_Unet weight")
ax_weight.axvline(T_s, color="tab:red", linestyle="--", label=f"T_s = {T_s}")
ax_weight.set(
    xlabel="diffusion timestep t",
    ylabel="hybrid mixing weight",
    ylim=(-0.02, 1.02),
    title="Hybrid mixing weights across diffusion steps",
)
ax_weight.legend()

ax_score.plot(t, covariance_score, color="tab:green", label=r"$\bar{\alpha}_t \lambda_{max}$")
ax_score.axhline(1.0, color="0.35", linestyle=":", label="target = 1")
ax_score.axvline(T_s, color="tab:red", linestyle="--", label=f"T_s = {T_s}")
ax_score.set(
    xlabel="diffusion timestep t",
    ylabel=r"$\bar{\alpha}_t \lambda_{max}$",
    title="Covariance criterion used to choose T_s",
)
ax_score.legend()

fig.suptitle(
    f"hybrid_ts_soft_lincomb: lambda_max={lambda_info['lambda_max']:.4g}, "
    f"T_s={T_s}, gate_tau={config['gate_tau']}",
    y=1.03,
)
fig.tight_layout()
plt.show()

pd.DataFrame({
    "t": [0, T_s, diffusion.num_timesteps - 1],
    "r_ode_lincomb": r_ode[[0, T_s, -1]],
    "r_cellunet": r_cellunet[[0, T_s, -1]],
    "alpha_bar_lambda_max": covariance_score[[0, T_s, -1]],
})